The purpose of this notebook is to process, create, and store metadata for full time series imagery. The outputs of this notebook are three CSV files which contain data from (1) the logs at the time of the EE pull, (2) attributes from the files themeslves, (3) attributes derived in this notebook the imagery itself, and (4) JSON metadata from earth engine. These files are what form the basis of the AWS Athena database

Metadata collection workflow:

1. Collect metadata from logs. This includes date the image was pulled; glims_id; crs; utm_zone; landsat image collection;

2. Collect metadata from metadata files for each landsat

3. Collect metadata from the image files. This includes the date the image was taken; the file size; 

4. Collect metadata from the images. This includes the image shape.

5. Join on filename

In [90]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm

sys.path.insert(0, os.path.join(os.path.expanduser("~"),"Desktop","projects", "GlacierView", "src","segmentation","helpers"))
import read
from rasterio.errors import RasterioIOError

In [91]:
data_label = "full_time_series_c02_t1_l2"
glacier_view_dir = os.path.join(os.path.expanduser('~'),"Desktop","projects","GlacierView")
# glaciers_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "landsat") # native storage
glaciers_dir = '/Volumes/T7/GlacierView/ee_landing_zone/full_time_series_c02_t1_l2/landsat' # external ssd
log_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "logs")
output_dir = os.path.join(glacier_view_dir, "src", "earth_engine", "data", "processed_metadata", data_label)
log_path =  os.path.join(log_dir,"log_geog_area_rollup_250.log")
glims_ids = [f for f in os.listdir(glaciers_dir) if not f.startswith('.')]

In [3]:
# # only run if you are adding to existing metadata with net new glims_ids
# net_new_df = pd.read_csv(os.path.join(output_dir,"net_new_glims_ids_for_segmentation.csv"))
# glims_ids = list(pd.Series(glims_ids)[pd.Series(glims_ids).isin(net_new_df.glims_id)])

### Step 1: Collect and clean log files from ee_pull_training

In [4]:
log_df = pd.read_csv(log_path, sep = ";", header = None)

In [5]:
log_df['ee_pull_date'] = log_df[0].str.split(",", expand = True)[0]
log_df['glims_id'] = log_df[0].str.split(",", expand = True)[1].str.split(":", expand = True)[3]
log_df['espg_crs'] = log_df[1].str.split(":", expand = True)[2]
log_df['cloud_cover'] = log_df[2].str.split(":", expand = True)[1]
log_df['cloud_cover_land'] = log_df[3].str.split(":", expand = True)[1]
log_df['scene_center_time'] = log_df[4].str.split(":", expand = True)[1]
log_df['image_quality'] = log_df[5].str.split(":", expand = True)[1]
log_df = log_df.drop([0,1,2,3,4,5,6], axis = 1)
log_df = log_df[log_df.glims_id.isin(glims_ids)]

In [6]:
log_df.to_csv(os.path.join(output_dir,"processed_logs_250.csv"), index = False)
log_df.head()

,ee_pull_date,glims_id,espg_crs,cloud_cover,cloud_cover_land,scene_center_time,image_quality
0,2024-09-11 17:40:30,G087896E49098N,32645,2,2,04,9
1,2024-09-11 17:40:33,G087896E49098N,32645,1,1,04,9
2,2024-09-11 17:41:19,G087896E49098N,32645,2,2,04,9
3,2024-09-11 17:41:23,G087896E49098N,32645,1,1,04,9
4,2024-09-11 17:41:26,G087896E49098N,32645,2,2,04,9


### Step 2: Collect file and image attributes (filename, file size in bytes, number of files)

In [92]:
file_attributes = []
image_attributes = []
for glims_id in tqdm(glims_ids):
    glacier_dir = os.path.join(glaciers_dir,glims_id)
    file_names = [f for f in os.listdir(glacier_dir) if "." in f and not f.startswith('.') and not f.endswith('.xml')]
    for file_name in file_names:
        #skip zip files
        if file_name.endswith(".tif"):
            #getting file attributes
            base_name = file_name.split(".")[0]
            glacier_path = os.path.join(glacier_dir,file_name)
            base_name_split = base_name.split("_")
            src_date = base_name_split[1]
            landsat = base_name_split[2]
            size_in_bytes = os.path.getsize(glacier_path)
            file_attributes.append((file_name, src_date, landsat, size_in_bytes))

            #getting image attributes
            with rasterio.open(os.path.join(glacier_dir,file_name)) as src:
                # Descriptions: https://rasterio.readthedocs.io/en/stable/api/rasterio.io.html#rasterio.io.DatasetReader
                height_in_pixels = src.height
                width_in_pixels = src.width
                left_bound = src.bounds.left
                right_bound = src.bounds.right
                top_bound = src.bounds.top
                bottom_bound = src.bounds.bottom
                num_of_bands = src.count
                epsg_code = int(src.crs.to_dict()['init'].split(":")[1])
                no_data_val = src.nodata
                pixel_width_res = src.res[0]
                pixel_height_res = src.res[1]

                try:
                    raster = src.read()
                except RasterioIOError: #strange error when reading some tiff files
                    continue
                zero_pixel_count = np.sum(raster == 0)
                no_data_pixel_count = np.sum(raster == no_data_val)
                negative_pixel_count = np.sum(raster < 0)
                image_attributes.append((file_name, height_in_pixels, width_in_pixels, left_bound,
                                         right_bound, top_bound, bottom_bound, num_of_bands, epsg_code,
                                        no_data_val, pixel_width_res, pixel_height_res, zero_pixel_count,
                                        no_data_pixel_count, negative_pixel_count))

100%|█████████████████████████████████████| 1101/1101 [2:37:01<00:00,  8.56s/it]


In [93]:
file_attributes_df = pd.DataFrame(file_attributes, columns = ["file_name", "src_date", "landsat", "file size in bytes"])
file_attributes_df.to_csv(os.path.join(output_dir,"file_attributes_250.csv"), index = False)
file_attributes_df.head()

,file_name,src_date,landsat,file size in bytes
0,G226067E58945N_2003-04-24_L7_C02_T1_L2_SR.tif,2003-04-24,L7,2272603
1,G226067E58945N_2008-04-20_L5_C02_T1_L2_SR.tif,2008-04-20,L5,2491620
2,G226067E58945N_2023-03-08_L7_C02_T1_L2_SR.tif,2023-03-08,L7,2356815
3,G226067E58945N_2010-04-11_L7_C02_T1_L2_SR.tif,2010-04-11,L7,1762328
4,G226067E58945N_2007-04-19_L7_C02_T1_L2_SR.tif,2007-04-19,L7,1731724


In [94]:
image_attributes_df = pd.DataFrame(image_attributes,
                                   columns = ["file_name", "height_in_pixels", "width_in_pixels", "left_bound",
                                             "right_bound", "top_bound", "bottom_bound", "num_of_bands", "epsg_code",
                                             "no_data_val", "pixel_width_res", "pixel_height_res", "zero_pixel_count",
                                             "no_data_pixel_count", "negative_pixel_count"])
image_attributes_df.to_csv(os.path.join(output_dir,"image_attributes_250.csv"), index = False)
image_attributes_df.head()

,file_name,height_in_pixels,width_in_pixels,left_bound,right_bound,top_bound,bottom_bound,num_of_bands,epsg_code,no_data_val,pixel_width_res,pixel_height_res,zero_pixel_count,no_data_pixel_count,negative_pixel_count
0,G226067E58945N_2003-04-24_L7_C02_T1_L2_SR.tif,460,311,554925.0,564255.0,6543405.0,6529605.0,19,32608,-2.147484e+09,30.0,30.0,7979,0,276
1,G226067E58945N_2008-04-20_L5_C02_T1_L2_SR.tif,460,311,554925.0,564255.0,6543405.0,6529605.0,19,32608,-2.147484e+09,30.0,30.0,47447,0,143336
2,G226067E58945N_2023-03-08_L7_C02_T1_L2_SR.tif,477,338,209595.0,219735.0,6553815.0,6539505.0,19,32609,-2.147484e+09,30.0,30.0,569712,0,560588
3,G226067E58945N_2010-04-11_L7_C02_T1_L2_SR.tif,460,311,554925.0,564255.0,6543405.0,6529605.0,19,32608,-2.147484e+09,30.0,30.0,265248,0,247819
4,G226067E58945N_2007-04-19_L7_C02_T1_L2_SR.tif,460,311,554925.0,564255.0,6543405.0,6529605.0,19,32608,-2.147484e+09,30.0,30.0,262730,0,248293


### Step 3: Collect JSON metadata from EE

In [23]:
# done_glims_ids = pd.read_csv(os.path.join(output_dir,"ee_metadata_250_part1.csv")).glims_id.unique()
# glims_ids = [glims_id for glims_id in glims_ids if glims_id not in done_glims_ids]

In [50]:
rows_to_insert = []
for glims_id in tqdm(glims_ids):
    glacier_dir = os.path.join(glaciers_dir, glims_id)
    if "meta_data" not in os.listdir(glacier_dir):
        continue
    meta_data_dir = os.path.join(glacier_dir, "meta_data")
    meta_data_file_names = os.listdir(meta_data_dir)
    for meta_data_file_name in meta_data_file_names:
        with open(os.path.join(meta_data_dir,meta_data_file_name)) as file:
            meta = file.read()
            for unformatted_meta in meta.split("\n")[1:-1]: #no data in first and last value
                truncated_meta = unformatted_meta.split(",\"")[1][:-1]
                meta_data = truncated_meta.replace("\'", "\"")
                meta_dict = json.loads(meta_data)
                meta_id_split = meta_dict['id'].split("_")

                landsat = "L" + str(int(meta_id_split[1][-2:]))
                date = meta_id_split[-1]
                year = date[:4]
                month = date[4:6]
                day = date[6:]
                file_name = f"{glims_id}_{year}-{month}-{day}_{landsat}_C02_T1_L2_SR.tif"
                ee_identifier = meta_dict['id']
                algorithm_source_surface_reflectance = meta_dict['properties']["ALGORITHM_SOURCE_SURFACE_REFLECTANCE"]
                try:
                    algorithm_source_surface_temperature = meta_dict['properties']["ALGORITHM_SOURCE_SURFACE_TEMPERATURE"]
                except KeyError:
                    algorithm_source_surface_temperature = ""
                cloud_cover = meta_dict['properties']["CLOUD_COVER"]
                cloud_cover_land = meta_dict['properties']["CLOUD_COVER_LAND"]
                collection_category = meta_dict['properties']["COLLECTION_CATEGORY"]
                data_source_air_temperature = meta_dict['properties']["DATA_SOURCE_AIR_TEMPERATURE"]
                data_source_elevation = meta_dict['properties']["DATA_SOURCE_ELEVATION"]
                data_source_ozone = meta_dict['properties']["DATA_SOURCE_OZONE"]
                data_source_pressure = meta_dict['properties']["DATA_SOURCE_PRESSURE"]
                try:
                    data_source_reanalysis = meta_dict['properties']["DATA_SOURCE_REANALYSIS"]
                except KeyError:
                    data_source_reanalysis = ""
                data_source_water_vapor = meta_dict['properties']["DATA_SOURCE_WATER_VAPOR"]
                date_product_generated = meta_dict['properties']["DATE_PRODUCT_GENERATED"]
                earth_sun_distance = meta_dict['properties']["EARTH_SUN_DISTANCE"]
                geometric_rmse_model = meta_dict['properties']["GEOMETRIC_RMSE_MODEL"]
                geometric_rmse_model_x = meta_dict['properties']["GEOMETRIC_RMSE_MODEL_X"]
                geometric_rmse_model_y = meta_dict['properties']["GEOMETRIC_RMSE_MODEL_Y"]
                ground_control_points_model = meta_dict['properties']["GROUND_CONTROL_POINTS_MODEL"]
                ground_control_points_version = meta_dict['properties']["GROUND_CONTROL_POINTS_VERSION"]
                l1_date_product_generated = meta_dict['properties']["L1_DATE_PRODUCT_GENERATED"]
                l1_landsat_product_id = meta_dict['properties']["L1_LANDSAT_PRODUCT_ID"]
                l1_processing_level = meta_dict['properties']["L1_PROCESSING_LEVEL"]
                l1_processing_software_version = meta_dict['properties']["L1_PROCESSING_SOFTWARE_VERSION"]
                landsat_product_id = meta_dict['properties']["LANDSAT_PRODUCT_ID"]
                landsat_scene_id = meta_dict['properties']["LANDSAT_SCENE_ID"]
                processing_level = meta_dict['properties']["PROCESSING_LEVEL"]
                processing_software_version = meta_dict['properties']["PROCESSING_SOFTWARE_VERSION"]
                scene_center_time = meta_dict['properties']["SCENE_CENTER_TIME"]
                sensor_id = meta_dict['properties']["SENSOR_ID"]
                spacecraft_id = meta_dict['properties']["SPACECRAFT_ID"]
                sun_azimuth = meta_dict['properties']["SUN_AZIMUTH"]
                sun_elevation = meta_dict['properties']["SUN_ELEVATION"]
                wrs_path = meta_dict['properties']["WRS_PATH"]
                wrs_row = meta_dict['properties']["WRS_ROW"]


                try:
                    image_quality = meta_dict['properties']['IMAGE_QUALITY']
                except KeyError:
                    image_quality = ""
                try:
                    image_quality_tirs = meta_dict['properties']['IMAGE_QUALITY_TIRS']
                except Exception:
                    image_quality_tirs = ""

                try:
                    image_quality_oli = meta_dict['properties']['IMAGE_QUALITY_OLI']
                except Exception:
                    image_quality_oli = ""
                try:
                    temperature_maximum_band_st_b6 = meta_dict['properties']['TEMPERATURE_MAXIMUM_BAND_ST_B6']
                except KeyError:
                    temperature_maximum_band_st_b6 = ""
                try:
                    temperature_minimum_band_st_b6 = meta_dict['properties']['TEMPERATURE_MINIMUM_BAND_ST_B6']
                except KeyError:
                    temperature_minimum_band_st_b6 = ""
                try:
                    ephemeris_type = meta_dict['properties']['EPHEMERIS_TYPE']
                except KeyError:
                    ephemeris_type = ""
                try:
                    data_source_tirs_stray_light_correction = meta_dict['properties']['DATA_SOURCE_TIRS_STRAY_LIGHT_CORRECTION']
                except KeyError:
                    data_source_tirs_stray_light_correction = ""
                try:
                    geometric_rmse_verify = meta_dict['properties']['GEOMETRIC_RMSE_VERIFY']
                except KeyError:
                    geometric_rmse_verify = ""
                try:
                    ground_control_points_verify = meta_dict['properties']['GROUND_CONTROL_POINTS_VERIFY']
                except KeyError:
                    ground_control_points_verify = ""
                try:
                    roll_angle = meta_dict['properties']['ROLL_ANGLE']
                except KeyError:
                    roll_angle = ""
                try:
                    target_wrs_path = meta_dict['properties']['TARGET_WRS_PATH']
                except KeyError:
                    target_wrs_path = ""
                try:
                    target_wrs_row = meta_dict['properties']['TARGET_WRS_ROW']
                except KeyError:
                    target_wrs_row = ""
                try:
                    temperature_maximum_band_st_b10 = meta_dict['properties']['TEMPERATURE_MAXIMUM_BAND_ST_B10']
                except KeyError:
                    temperature_maximum_band_st_b10 = ""
                try:
                    temperature_minimum_band_st_b10 = meta_dict['properties']['TEMPERATURE_MINIMUM_BAND_ST_B10']
                except KeyError:
                    temperature_minimum_band_st_b10 = ""
                try:
                    tirs_ssm_model = meta_dict['properties']['TIRS_SSM_MODEL']
                except KeyError:
                    tirs_ssm_model = ""


                ee_meta_data = {
                                "file_name": file_name,
                                "src_date": f"{year}-{month}-{day}",
                                "ee_identifier": ee_identifier,
                                "glims_id": glims_id,
                                "algorithm_source_surface_reflectance": algorithm_source_surface_reflectance,
                                "algorithm_source_surface_temperature": algorithm_source_surface_temperature,
                                "cloud_cover": cloud_cover,
                                "cloud_cover_land": cloud_cover_land,
                                "collection_category": collection_category,
                                "data_source_air_temperature": data_source_air_temperature,
                                "data_source_elevation": data_source_elevation,
                                "data_source_ozone": data_source_ozone,
                                "data_source_pressure": data_source_pressure,
                                "data_source_reanalysis": data_source_reanalysis,
                                "data_source_water_vapor": data_source_water_vapor,
                                "date_product_generated": date_product_generated,
                                "earth_sun_distance": earth_sun_distance,
                                "geometric_rmse_model": geometric_rmse_model,
                                "geometric_rmse_model_x": geometric_rmse_model_x,
                                "geometric_rmse_model_y": geometric_rmse_model_y,
                                "ground_control_points_model": ground_control_points_model,
                                "ground_control_points_version": ground_control_points_version,
                                "l1_date_product_generated": l1_date_product_generated,
                                "l1_landsat_product_id": l1_landsat_product_id,
                                "l1_processing_level": l1_processing_level,
                                "l1_processing_software_version": l1_processing_software_version,
                                "landsat_product_id": landsat_product_id,
                                "landsat_scene_id": landsat_scene_id,
                                "processing_level": processing_level,
                                "processing_software_version": processing_software_version,
                                "scene_center_time": scene_center_time,
                                "sensor_id": sensor_id,
                                "spacecraft_id": spacecraft_id,
                                "sun_azimuth": sun_azimuth,
                                "sun_elevation": sun_elevation,
                                "wrs_path": wrs_path,
                                "wrs_row": wrs_row,
                                "image_quality": image_quality,
                                "image_quality_oli": image_quality_oli ,
                                "image_quality_tirs": image_quality_tirs ,
                                "temperature_maximum_band_st_b6": temperature_maximum_band_st_b6 ,
                                "temperature_minimum_band_st_b6": temperature_minimum_band_st_b6 ,
                                "ephemeris_type": ephemeris_type ,
                                "data_source_tirs_stray_light_correction": data_source_tirs_stray_light_correction ,
                                "geometric_rmse_verify": geometric_rmse_verify ,
                                "ground_control_points_verify": ground_control_points_verify ,
                                "roll_angle": roll_angle ,
                                "target_wrs_path": target_wrs_path ,
                                "target_wrs_row": target_wrs_row ,
                                "temperature_maximum_band_st_b10": temperature_maximum_band_st_b10 ,
                                "temperature_minimum_band_st_b10": temperature_minimum_band_st_b10 ,
                                "tirs_ssm_model": tirs_ssm_model
                              }
                rows_to_insert.append(ee_meta_data)

100%|███████████████████████████████████████| 1101/1101 [01:22<00:00, 13.28it/s]


In [108]:
ee_meta_data_df = pd.DataFrame(rows_to_insert)

# sometimes EE provides more than one metadata per image, so we will dedup arbitrarily
ee_meta_data_df[~ee_meta_data_df.duplicated(subset='file_name', keep = "first")].sort_values(by="file_name").to_csv(os.path.join(output_dir,"ee_metadata_250.csv"), index = False)

#### Why do we have duplicate rows per file name for ee metadata?

In [74]:
print(f"Unique file names: {len(ee_meta_data_df.file_name.unique())}")
print(f"Row count: {ee_meta_data_df.shape[0]}")

Unique file names: 507762
Row count: 592897


In [87]:
ee_meta_data_df[ee_meta_data_df.duplicated(subset="file_name")].sort_values(by="file_name").head()

,file_name,src_date,ee_identifier,glims_id,algorithm_source_surface_reflectance,algorithm_source_surface_temperature,cloud_cover,cloud_cover_land,collection_category,data_source_air_temperature,...,ephemeris_type,data_source_tirs_stray_light_correction,geometric_rmse_verify,ground_control_points_verify,roll_angle,target_wrs_path,target_wrs_row,temperature_maximum_band_st_b10,temperature_minimum_band_st_b10,tirs_ssm_model
570920,G007068E45470N_2013-11-12_L8_C02_T1_L2_SR.tif,2013-11-12,LANDSAT/LC08/C02/T1_L2/LC08_195029_20131112,G007068E45470N,LaSRC_1.5.0,st_1.3.0,5.73,5.86,T1,MODIS,...,,TIRS,4.091,149,-0.001,195,29,372.999941,149.003418,ACTUAL
570921,G007068E45470N_2013-11-28_L8_C02_T1_L2_SR.tif,2013-11-28,LANDSAT/LC08/C02/T1_L2/LC08_195029_20131128,G007068E45470N,LaSRC_1.5.0,st_1.3.0,5.20,5.32,T1,MODIS,...,,TIRS,4.758,120,-0.001,195,29,372.999941,149.003418,ACTUAL
570922,G007068E45470N_2013-12-30_L8_C02_T1_L2_SR.tif,2013-12-30,LANDSAT/LC08/C02/T1_L2/LC08_195029_20131230,G007068E45470N,LaSRC_1.5.0,st_1.3.0,0.58,0.59,T1,MODIS,...,,TIRS,5.847,137,-0.001,195,29,372.999941,149.003418,ACTUAL
570924,G007068E45470N_2014-06-08_L8_C02_T1_L2_SR.tif,2014-06-08,LANDSAT/LC08/C02/T1_L2/LC08_195029_20140608,G007068E45470N,LaSRC_1.5.0,st_1.3.0,9.04,8.94,T1,MODIS,...,,TIRS,2.486,153,-0.001,195,29,372.999941,149.003418,ACTUAL
570926,G007068E45470N_2014-10-30_L8_C02_T1_L2_SR.tif,2014-10-30,LANDSAT/LC08/C02/T1_L2/LC08_195029_20141030,G007068E45470N,LaSRC_1.5.0,st_1.3.0,1.17,1.19,T1,MODIS,...,,TIRS,3.574,154,-0.001,195,29,372.999941,149.003418,ACTUAL


In [59]:
# ee_meta_data_df = pd.DataFrame({
#                                 "file_name":[],
#                                 "src_date": [],
#                                 "ee_identifier":[],
#                                 "glims_id":[],
#                                 "algorithm_source_surface_reflectance": [],
#                                 "algorithm_source_surface_temperature": [],
#                                 "cloud_cover": [],
#                                 "cloud_cover_land": [],
#                                 "collection_category": [],
#                                 "data_source_air_temperature": [],
#                                 "data_source_elevation": [],
#                                 "data_source_ozone": [],
#                                 "data_source_pressure": [],
#                                 "data_source_reanalysis": [],
#                                 "data_source_water_vapor": [],
#                                 "date_product_generated": [],
#                                 "earth_sun_distance": [],
#                                 "geometric_rmse_model": [],
#                                 "geometric_rmse_model_x": [],
#                                 "geometric_rmse_model_y": [],
#                                 "ground_control_points_model": [],
#                                 "ground_control_points_version": [],
#                                 "l1_date_product_generated": [],
#                                 "l1_landsat_product_id": [],
#                                 "l1_processing_level": [],
#                                 "l1_processing_software_version": [],
#                                 "landsat_product_id": [],
#                                 "landsat_scene_id": [],
#                                 "processing_level": [],
#                                 "processing_software_version": [],
#                                 "scene_center_time": [],
#                                 "sensor_id": [],
#                                 "spacecraft_id": [],
#                                 "sun_azimuth": [],
#                                 "sun_elevation": [],
#                                 "wrs_path": [],
#                                 "wrs_row": [],
#                                 "image_quality": [],
#                                 "image_quality_oli": [],
#                                 "image_quality_tirs": [],
#                                 "temperature_maximum_band_st_b6": [],
#                                 "temperature_minimum_band_st_b6": [],
#                                 "ephemeris_type": [],
#                                 "data_source_tirs_stray_light_correction": [],
#                                 "geometric_rmse_verify": [],
#                                 "ground_control_points_verify": [],
#                                 "roll_angle": [],
#                                 "target_wrs_path": [],
#                                 "target_wrs_row": [],
#                                 "temperature_maximum_band_st_b10": [],
#                                 "temperature_minimum_band_st_b10": [],
#                                 "tirs_ssm_model": []
#                                })